In [1]:
import base
import observer
import attrib
import stores
import numpy
import pandas
import geopandas
import plotly.express
import folium
from folium import Choropleth
from folium.plugins import TimestampedGeoJson
import shapely.geometry
from datetime import datetime
#import matplotlib.cm as cm
from matplotlib.colors import rgb2hex
import matplotlib.pyplot as plt  # Add this import statement
from IPython.display import display, HTML
from matplotlib import colormaps as cm
import imageio
import os
from selenium import webdriver
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [2]:
concentrations = base.Input(
    "ConcentrationInFruits",
    (attrib.Class(numpy.ndarray), attrib.Unit("mg/kg"), attrib.Scales("space/base_geometry, time/day")),
    observer.ConsoleObserver(print_output=True),
    base.Output(
        "Dpu/ConcentrationInFruits",
        stores.X3dfStore(r"E:\Bayer\bay.18\models\xKnowRes230822\run\TestRun_xKnowRes\mcs\X3UDQO3DQEVSMR85MU\store", mode="r")
    )
).read()

NOTE  Input ConcentrationInFruits is missing a detailed description

OK    ConcentrationInFruits:ClassChecker:GetValues

      Values are of type <class 'numpy.ndarray'>

OK    ConcentrationInFruits:UnitChecker:GetValues

      Values have unit mg/kg

OK    ConcentrationInFruits:ScalesChecker:GetValues

      Values have scales space/base_geometry, time/day



In [3]:
field_geometries = geopandas.GeoDataFrame(
    geometry=geopandas.GeoSeries.from_wkb(concentrations.geometries[0].get_values()),
    crs="EPSG:3857"
).to_crs(crs="EPSG:4326")
field_geometries["field_idx"] = field_geometries.reset_index().index

In [4]:
geo_df = field_geometries.merge(
    pandas.DataFrame(
       numpy.c_[
           numpy.argwhere(concentrations.values > 0),
           concentrations.values[concentrations.values > 0]
       ],
       columns=("field_idx", "time_idx", "concentration")
    ),
    on="field_idx"
)

In [5]:
geo_df["date"] = pandas.to_datetime(concentrations.offsets[1]) + pandas.to_timedelta(geo_df["time_idx"], unit="D")
geo_df["date"] = geo_df["date"].dt.strftime("%Y-%m-%d")
geo_df.to_file("f:/concentrations.shp")

C:\Users\JillianLaRoe\AppData\Local\Temp\ipykernel_18732\4255526484.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  geo_df.to_file("f:/concentrations.shp")


In [35]:
geo_df = geopandas.read_file("f:/concentrations.shp")

In [7]:
m = folium.Map(location=[50.383, 8.659], zoom_start=14,
               zoom_control=False,
               scrollWheelZoom=False,
               dragging=False)

In [8]:
# color map and normalization
cmap = cm.get_cmap('YlGnBu')  
norm = plt.Normalize(vmin=geo_df['concentrat'].min(), vmax=geo_df['concentrat'].max())
print(norm)
min_concentration = geo_df['concentrat'].min()
max_concentration = geo_df['concentrat'].max()



print(max_concentration)
type(max_concentration)

color_swatches = ''
#for value in range(mic, mac + 1):
for value in numpy.arange(min_concentration, max_concentration, 0.02):
    color = rgb2hex(cmap(norm(value)))
    color_swatches += f'<div style="background-color: {color}; width: 20px; height: 20px; display: inline-block;"></div>'
    
print(color)

0.187270351901238
#102369


In [9]:
# Create Choropleth layer
choropleth = folium.Choropleth(
    geo_data=geo_df,
    name='choropleth',
    data=geo_df,
    columns=['time_idx', 'concentrat'],
    key_on='feature.properties.time_idx',
    fill_color='YlGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Concentration',
)


In [10]:
print(numpy.arange(min_concentration, max_concentration, 0.02))

[3.86219300e-09 2.00000039e-02 4.00000039e-02 6.00000039e-02
 8.00000039e-02 1.00000004e-01 1.20000004e-01 1.40000004e-01
 1.60000004e-01 1.80000004e-01]


In [11]:
features = []

for _, row in geo_df.iterrows():
    feature = {
        'type': 'Feature',
        'geometry': row['geometry'].__geo_interface__,
        'properties': {
            'time': row['date'],
            'popup': row['concentrat'],
            'style': {
                'color': rgb2hex(cmap(norm(row['concentrat'])))
            }
        }
    }
    features.append(feature) 

In [12]:
TimestampedGeoJson({'type': 'FeatureCollection', 'features': features}, period="P1D",  transition_time=30000, max_speed=5000, min_speed=4000, date_options="YYYY/MM/DD", add_last_point=False,loop=False).add_to(m)

In [13]:
# Create custom HTML legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index:1000; background-color: white; padding: 10px; border: 2px solid grey; font-size: 14px;">
    <div style="background-color: white; color: black; padding: 5px; border: 2px solid grey;">Legend</div>
    <div style="background-color: white; color: black; padding: 5px; border: 2px solid grey;">Min: {}</div>
    <div style="background-color: white; color: black; padding: 5px; border: 2px solid grey;">Max: {}</div>
    <div style="background-color: white; color: black; padding: 5px; border: 2px solid grey;">Color Ramp: {}</div>
</div>
</div>
'''.format(min_concentration, max_concentration, color_swatches)

# Add legend to map
m.get_root().html.add_child(folium.Element(legend_html))

In [ ]:
html_map = m._repr_html_()
display(HTML(html_map))

In [19]:
# Display map
m.save('f:/map_with_time_animation4.html')

In [20]:
# Dir to save frames
frames_directory = 'f:/frames2'
os.makedirs(frames_directory, exist_ok=True)



In [46]:
day_count = max(geo_df['time_idx']) - min(geo_df['time_idx']) + 1
day_count

836.0

In [47]:
total_frames_to_capture = day_count

html_file_path = 'f:/map_with_time_animation4.html'

# headless browser instance (no GUI)
options = webdriver.ChromeOptions()
options.add_argument('headless')
browser = webdriver.Chrome(options=options)

# Open HTML file
browser.get('file://' + html_file_path)

# Define output dir for screenshots
screenshot_dir = 'f:/frames2'
os.makedirs(screenshot_dir, exist_ok=True)



# Capture screenshot for each frame
frame_count = 0
while frame_count < total_frames_to_capture:
    screenshot_path = os.path.join(screenshot_dir, f'frame_{frame_count:04d}.png')
    browser.save_screenshot(screenshot_path)

    # Wait for "next frame" to fully display
    wait = WebDriverWait(browser, 4)

    try:
        next_button = browser.find_element(By.XPATH,'/html/body/div[2]/div[2]/div[3]/div/a[3]')  # element ID
        next_button.click()
        frame_count += 1
        time.sleep(6)  # Adjust sleep time if needed
    except Exception as e:
        print(f"Error: {e}")
        break  

browser.quit()  

print(f'{frame_count} frames captured.')

836 frames captured.


In [48]:


# Dir of screenshots 
screenshot_dir = 'f:/frames2'

# List of screenshots 
screenshot_files = [os.path.join(frames_directory, filename) for filename in os.listdir(frames_directory) if filename.startswith('frame_')]

# Sort the screenshot files to ensure correct order
screenshot_files.sort()


# Output MP4 file path
output_video_path = 'f:/frames2/test_animation_20fps_v2.mp4'

# Define frames per second (fps) for video
fps = 20  

# Create MP4 video from frames
with imageio.get_writer(output_video_path, mode='I', fps=fps) as writer:
    for screenshot_file in screenshot_files:
        frame = imageio.imread(screenshot_file)
        writer.append_data(frame)

print(f'Video saved as: {output_video_path}')


C:\Users\JillianLaRoe\AppData\Local\Temp\ipykernel_18732\2116222413.py:20: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imageio.imread(screenshot_file)
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (800, 600) to (800, 608) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Video saved as: f:/frames2/test_animation_20fps_v2.mp4


In [ ]:

# map_html = os.path.join(frames_directory, 'map.html')
# m.save(map_html)

# # Capture frames using imageio
# frame_format = os.path.join(frames_directory, 'frame_{}.png')

# for i in range(len(features)):
#     # Open the saved HTML file in a web browser
#     m = folium.Map(location=[50.383, 8.659], zoom_start=14)
#     folium.IFrame(html=open(map_html, 'r').read(), width=800, height=600).add_to(m)
    
#     # Capture the frame as an image
#     frame_path = frame_format.format(i)
#     m.save(frame_path)

# # Clean up the temporary HTML file
# os.remove(map_html)



In [ ]:

# # List the frame files in the directory
# frame_files = sorted([os.path.join(frames_directory, filename) for filename in os.listdir(frames_directory)])



# # Create the MP4 video from frames using imageio
# with imageio.get_writer(output_video_path, mode='I', fps=fps) as writer:
#     for frame_file in frame_files:
#         frame = imageio.imread(frame_file)
#         writer.append_data(frame)

# print(f'Video saved as: {output_video_path}')

In [82]:
# # List the unique dates from the features
# unique_dates = set()
# for feature in features:
#     if 'date' in feature:
#         unique_dates.add(feature['date'])

# print(unique_dates)

# unique_dates_col = geo_df['date'].unique()
# unique_dates = unique_dates_col.tolist()

# N = 5
 
# # using list slicing
# # Get first N elements from list
# res = unique_dates[:N]

# # print result
# print("The first N elements of list are : " + str(res))

set()
The first N elements of list are : ['2014-04-25', '2014-04-26', '2014-04-27', '2014-04-28', '2014-04-29']


In [ ]:
# # Create the MP4 video from unique frames using imageio
# with imageio.get_writer(output_video_path, mode='I', fps=fps) as writer:
#     for date in unique_dates:
#         # Open the saved HTML file in a web browser for the specific date
#         m = folium.Map(location=[50.383, 8.659], zoom_start=14)
#         iframe_html = folium.IFrame(html=open(map_html, 'r').read(), width=800, height=600)
#         folium.Marker([50.383, 8.659], icon=folium.DivIcon(html=iframe_html)).add_to(m)
        
#         # Capture the frame as an image
#         frame_path = os.path.join(frames_directory, 'frame_{}.png'.format(date))
#         m.save(frame_path)

# #print(f'Video saved as: {output_video_path}')

In [78]:
# features = []

# # Extract 'when' values and add features to the list
# for date, feature in features:  # Adjust your data structure here
#     feature = {
#         'type': 'Feature',
#         'geometry': feature_data['geometry'],  # Adjust this based on your data
#         'properties': {
#             'popup': feature_data['concentration']  # Adjust this based on your data
#         }
#     }
#     features.append({'when': date, 'feature': feature})